<a href="https://colab.research.google.com/github/ever1318-cpu/Apartment_Defect_AI/blob/main/Step02_NOVAPro_%ED%8C%90%EB%B3%84_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 01 — 데이터셋 만들기 (Colab 실행용)

DB에서 하자(defect) 자료를 읽어 메타데이터를 JSON으로 저장합니다.

**출력 파일**
- `dataset_nova.json` — 표본 하자 건 메타데이터 + 사진 URL + 후보 라벨
- `defect_ids_used_nova.json` — 표본 defect_id 목록 (교차검증/재현용)

> ⚠️ 이 노트북은 실서비스 DB에 직접 접속합니다. 비밀번호는 하드코딩하지 않고, 실행할 때마다 안전하게 입력받도록 구성했습니다.

## 1. 패키지 설치

In [59]:
!pip install psycopg2-binary -q

## 2. 라이브러리 임포트

In [60]:
import json
from pathlib import Path
from getpass import getpass

import psycopg2

## 3. DB 접속 정보

비밀번호는 코드에 남기지 않고, 실행 시점에 입력받습니다.
(같은 값을 매번 입력하기 번거로우면 아래 `DB_PASSWORD` 줄의 주석을 풀고 직접 넣어도 됩니다 — 다만 노트북을 공유/저장할 계획이면 권장하지 않습니다.)

In [61]:
DB_HOST = "w-backupdb.c9u0e882q8zp.ap-northeast-2.rds.amazonaws.com"
DB_PORT = 5432
DB_NAME = "BackupDB"
DB_USER = "TruePostgres"

# 실행할 때마다 비밀번호를 입력받습니다 (화면에 노출되지 않음)
#DB_PASSWORD = getpass("DB 비밀번호 입력: ")
DB_PASSWORD = "true2026!!"

DB_CONFIG = {
    "host": DB_HOST,
    "port": DB_PORT,
    "dbname": DB_NAME,
    "user": DB_USER,
    "password": DB_PASSWORD,
}


def get_conn():
    return psycopg2.connect(**DB_CONFIG)

## 4. 추출 설정

In [62]:
#SITE_CODE = "P0256D"  # 울산 다운1차
SITE_CODE = None      # 전체 단지로 확장하려면 None
# SITE_CODE = "P0165D"  # 과천

# 하자 사진 저장 S3
S3_BASE_URL = "https://wmcsm-defect-file.s3.ap-northeast-2.amazonaws.com/"

# --- 사용자 설정: 추출할 데이터 건수 ---
USER_EXTRACTION_COUNT = 2000  # 원하는 추출 수 (예: 100, 1000, 10000)
# -----------------------------------

SAMPLE_LIMIT = USER_EXTRACTION_COUNT
RANDOM_SEED = 0.4207          # 기존 표본과 최대한 겹치도록 동일 seed 유지
MAX_BEFORE_IMAGES = 3

OUT_PATH = Path("dataset_nova.json")
IDS_PATH = Path("defect_ids_used_nova.json")

## 5. 모집단 규모 확인

In [63]:
def count_eligible(cur):
    cur.execute(
        """
        WITH EligibleDefects AS (
            SELECT sd.id
            FROM site_defect sd
            JOIN defect_ho dh ON dh.id = sd.ho_id
            JOIN defect_dong dd ON dd.id = dh.dong_id
            JOIN defect_site ds ON ds.id = dd.site_id
            LEFT JOIN site_defect_file sdf ON sdf.defect_id = sd.id
            WHERE (%(site_code)s IS NULL OR ds.code = %(site_code)s)
              AND sd.type = 'QUALITY'
            GROUP BY sd.id
            HAVING COUNT(sdf.id) FILTER (
                WHERE sdf.full_path IS NOT NULL
                  AND TRIM(sdf.full_path) <> ''
                  AND UPPER(TRIM(sdf.file_type)) = 'BEFORE'
            ) > 0
        )
        SELECT COUNT(id) FROM EligibleDefects
        """,
        {"site_code": SITE_CODE},
    )
    result = cur.fetchone()
    return result[0] if result else 0

## 6. 실/부위/상세부위/공종/하자원인 후보 목록 (계층 매핑)

In [64]:
def load_hierarchy_maps(cur):
    base_from = """
        FROM site_defect sd
        JOIN defect_ho dh ON dh.id = sd.ho_id
        JOIN defect_dong dd ON dd.id = dh.dong_id
        JOIN defect_site ds ON ds.id = dd.site_id
        LEFT JOIN site_site_defect_item di ON di.id = sd.site_defect_item_id
        LEFT JOIN site_site_part_detail_map dm ON dm.id = di.site_site_part_detail_map_id
        LEFT JOIN site_site_area_part_map pm ON pm.id = dm.site_area_part_map_id
        LEFT JOIN defect_area da ON da.id = pm.area_id
        LEFT JOIN defect_part dp ON dp.id = pm.part_id
    """
    base_where = "WHERE (%(site_code)s IS NULL OR ds.code = %(site_code)s) AND sd.type = 'QUALITY'"
    params = {"site_code": SITE_CODE}

    cur.execute(f"SELECT DISTINCT da.name, dp.name {base_from} {base_where}", params)
    area_to_parts = {}
    for area, part in cur.fetchall():
        if area and part:
            area_to_parts.setdefault(area, set()).add(part)

    cur.execute(
        f"""SELECT DISTINCT da.name, dp.name, pdd.name {base_from}
            LEFT JOIN defect_part_detail pdd ON pdd.id = dm.part_detail_id {base_where}""",
        params,
    )
    area_part_to_details = {}
    for area, part, detail in cur.fetchall():
        if area and part and detail:
            area_part_to_details.setdefault((area, part), set()).add(detail)

    cur.execute(
        f"""SELECT DISTINCT da.name, dp.name, wk.name {base_from}
            LEFT JOIN defect_work_kind wk ON wk.id = sd.work_kind_id {base_where}""",
        params,
    )
    area_part_to_workkinds = {}
    for area, part, wk in cur.fetchall():
        if area and part and wk:
            area_part_to_workkinds.setdefault((area, part), set()).add(wk)

    # 하자원인(하자종류)은 실/부위와 직접적인 계층관계가 없는 별도 분류축이므로
    # 위치 기준으로 좁히지 않고 단지 전체 후보를 그대로 준다.
    cur.execute(
        f"""SELECT DISTINCT dc.name
            FROM site_defect sd
            JOIN defect_ho dh ON dh.id = sd.ho_id
            JOIN defect_dong dd ON dd.id = dh.dong_id
            JOIN defect_site ds ON ds.id = dd.site_id
            LEFT JOIN defect_cause dc ON dc.id = sd.cause_id
            {base_where}""",
        params,
    )
    cause_list = sorted(r[0] for r in cur.fetchall() if r[0])

    return {
        "all_areas": sorted(area_to_parts.keys()),
        "area_to_parts": area_to_parts,
        "area_part_to_details": area_part_to_details,
        "area_part_to_workkinds": area_part_to_workkinds,
        "cause_list": cause_list,
    }


def resolve_candidates(defect, maps):
    org_area, org_part, org_detail = defect.get("실"), defect.get("부위"), defect.get("상세부위")

    area_candidates = maps["all_areas"]
    part_candidates = sorted(maps["area_to_parts"].get(org_area, set())) or sorted(
        {p for ps in maps["area_to_parts"].values() for p in ps}
    )
    detail_candidates = sorted(maps["area_part_to_details"].get((org_area, org_part), set()))
    if not detail_candidates and org_detail:
        detail_candidates = [org_detail]
    workkind_candidates = sorted(maps["area_part_to_workkinds"].get((org_area, org_part), set())) or sorted(
        {wk for wks in maps["area_part_to_workkinds"].values() for wk in wks}
    )
    return area_candidates, part_candidates, detail_candidates, workkind_candidates

## 7. 하자 건 + 사진 + 하자원인 포함 조회

In [65]:
def fetch_defects(cur, limit, seed, pool_limit):
    order_expr = "md5(sd.id::text || %(seed_salt)s)" if seed is not None else "random()"
    sql = f"""
        WITH candidate_ids AS (
            SELECT sd.id
            FROM site_defect sd
            JOIN defect_ho dh ON dh.id = sd.ho_id
            JOIN defect_dong dd ON dd.id = dh.dong_id
            JOIN defect_site ds ON ds.id = dd.site_id
            LEFT JOIN site_defect_file sdf ON sdf.defect_id = sd.id
            WHERE (%(site_code)s IS NULL OR ds.code = %(site_code)s)
              AND sd.type = 'QUALITY'
            GROUP BY sd.id
            HAVING COUNT(sdf.id) FILTER (
                WHERE sdf.full_path IS NOT NULL
                  AND TRIM(sdf.full_path) <> ''
                  AND UPPER(TRIM(sdf.file_type)) = 'BEFORE'
            )
            > 0
            ORDER BY {order_expr}
            LIMIT %(pool_limit)s
        )
        SELECT
            sd.id AS defect_id,
            ds.name AS 단지명,
            dd.name AS 동, dh.name AS 호, da.name AS 실,
            dp.name AS 부위, pdd.name AS 상세부위,
            sd.requirement AS 고객민원내용,
            dc.name AS 원본_하자원인,
            wk.name AS 원본_공종,
            json_agg(
                json_build_object(
                    'file_type', sdf.file_type,
                    'full_path', sdf.full_path,
                    'url', %(base_url)s || ltrim(sdf.full_path, '/')
                ) ORDER BY sdf.file_type, sdf.id
            ) FILTER (
                WHERE sdf.full_path IS NOT NULL
                  AND TRIM(sdf.full_path) <> ''
                  AND UPPER(TRIM(sdf.file_type)) = 'BEFORE'
            ) AS 사진목록
        FROM site_defect sd
        JOIN candidate_ids ci ON ci.id = sd.id
        JOIN defect_ho dh ON dh.id = sd.ho_id
        JOIN defect_dong dd ON dd.id = dh.dong_id
        JOIN defect_site ds ON ds.id = dd.site_id
        LEFT JOIN site_site_defect_item di ON di.id = sd.site_defect_item_id
        LEFT JOIN site_site_part_detail_map dm ON dm.id = di.site_site_part_detail_map_id
        LEFT JOIN site_site_area_part_map pm ON pm.id = dm.site_area_part_map_id
        LEFT JOIN defect_area da ON da.id = pm.area_id
        LEFT JOIN defect_part dp ON dp.id = pm.part_id
        LEFT JOIN defect_part_detail pdd ON pdd.id = dm.part_detail_id
        LEFT JOIN defect_cause dc ON dc.id = sd.cause_id
        LEFT JOIN defect_work_kind wk ON wk.id = sd.work_kind_id
        LEFT JOIN site_defect_file sdf ON sdf.defect_id = sd.id
        GROUP BY sd.id, ds.name, dd.name, dh.name, da.name, dp.name, pdd.name,
                 sd.requirement, dc.name, wk.name
        HAVING count(sdf.id) FILTER (
            WHERE sdf.full_path IS NOT NULL
              AND TRIM(sdf.full_path) <> ''
              AND UPPER(TRIM(sdf.file_type)) = 'BEFORE'
        ) > 0
        ORDER BY sd.id
        LIMIT %(limit)s
    """
    params = {
        "base_url": S3_BASE_URL,
        "site_code": SITE_CODE,
        "pool_limit": pool_limit,
        "limit": limit,
    }
    if seed is not None:
        params["seed_salt"] = str(seed)
    cur.execute(sql, params)
    cols = [d[0] for d in cur.description]
    defects = []
    for row in cur.fetchall():
        d = dict(zip(cols, row))
        photos = d.pop("사진목록") or []
        d["photos_before"] = [p for p in photos if p.get("file_type") == "BEFORE"]
        defects.append(d)
    return defects

## 8. 실행 (메인 파이프라인)

In [66]:
def main():
    conn = get_conn()
    cur = conn.cursor()

    total_eligible = count_eligible(cur)
    print(f"모집단(사진 보유 QUALITY 하자, SITE_CODE={SITE_CODE}) 규모: {total_eligible:,}건")
    if total_eligible < SAMPLE_LIMIT:
        print(f"[안내] 모집단이 목표({SAMPLE_LIMIT:,}건)보다 적습니다. 있는 만큼({total_eligible:,}건) 전부 추출합니다.")

    pool_limit = min(SAMPLE_LIMIT * 5, total_eligible) if total_eligible else SAMPLE_LIMIT * 5

    maps = load_hierarchy_maps(cur)
    print(f"후보 목록 로드 완료: 실 {len(maps['all_areas'])}종 / 하자원인 {len(maps['cause_list'])}종")

    defects = fetch_defects(cur, SAMPLE_LIMIT, seed=RANDOM_SEED, pool_limit=pool_limit)
    cur.close()
    conn.close()
    print(f"하자 건 {len(defects)}건 조회 완료 (사진 보유 건 기준)")

    records = []
    skipped_no_photo = 0
    for d in defects:
        photos = [p["url"] for p in d["photos_before"][:MAX_BEFORE_IMAGES]]
        if not photos:
            skipped_no_photo += 1
            continue
        area_c, part_c, detail_c, workkind_c = resolve_candidates(d, maps)
        records.append(
            {
                "defect_id": d["defect_id"],
                "단지명": d["단지명"], "동": d["동"], "호": d["호"],
                "실": d["실"], "부위": d["부위"], "상세부위": d["상세부위"],
                "고객민원내용": d["고객민원내용"],
                "원본_하자원인": d.get("원본_하자원인"),
                "원본_공종": d.get("원본_공종"),
                "photos_before_urls": photos,
                "candidates": {
                    "실": area_c, "부위": part_c, "상세부위": detail_c,
                    "공종": workkind_c, "하자원인": maps["cause_list"],
                },
            }
        )

    OUT_PATH.write_text(json.dumps(records, ensure_ascii=False, indent=2), encoding="utf-8")
    IDS_PATH.write_text(json.dumps([r["defect_id"] for r in records], ensure_ascii=False), encoding="utf-8")
    print(f"\n내보내기 완료: {len(records)}건 (사진 없어서 제외: {skipped_no_photo}건)")
    print(f"저장: {OUT_PATH.resolve()}")
    print("저장된 JSON 파일을 이용하여 Nova pro 로 분석 진행하세요")


main()

모집단(사진 보유 QUALITY 하자, SITE_CODE=None) 규모: 151,827건
후보 목록 로드 완료: 실 70종 / 하자원인 11종
하자 건 2000건 조회 완료 (사진 보유 건 기준)

내보내기 완료: 2000건 (사진 없어서 제외: 0건)
저장: /content/dataset_nova.json
저장된 JSON 파일을 이용하여 Nova pro 로 분석 진행하세요


## 9. (선택) 결과를 구글 드라이브에 저장

Colab 세션이 끝나면 로컬 파일이 사라집니다. 결과를 보존하려면 드라이브에 마운트 후 복사하세요.

In [67]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
SAVE_DIR = "/content/drive/MyDrive/dataset_nova_outputs"  # 원하는 경로로 수정
import os
os.makedirs(SAVE_DIR, exist_ok=True)
shutil.copy(OUT_PATH, SAVE_DIR)
shutil.copy(IDS_PATH, SAVE_DIR)
print(f"드라이브에 저장 완료: {SAVE_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
드라이브에 저장 완료: /content/drive/MyDrive/dataset_nova_outputs


---
# Step 02 (재정의) — AWS Bedrock Nova Pro 판독 · 정답 병기 비교

박태홍님 작성 로직 기준으로 재구성했습니다. 이전 버전과 달라진 점:
- Bearer API 키 대신 **AWS IAM 자격증명(boto3)** 사용 — Colab Secrets 등록 권장
- 실/부위/상세부위/하자원인/공종 5개 항목 판독 + **상세부위는 정확일치·부분일치 둘 다 계산**
- 스로틀링(Throttling) 자동 재시도
- 결과를 `원본_X` / `AI_X` **병기 컬럼**으로 정리 → 엑셀(`NovaPro_분석최종결과.xlsx`)로 저장
- 빈도 상위 N개 클래스만 따로 본 정확도(불균형 클래스 영향 확인용)도 함께 계산

> ⚠️ **보안 주의**: AWS 자격증명은 절대 셀에 하드코딩하지 마세요. Colab 좌측 열쇠 아이콘(🔑) → Secrets에
> `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`(필요시 `AWS_SESSION_TOKEN`)를 등록해두면 자동으로 읽어옵니다.

## 11. 패키지 설치 및 임포트

In [68]:
!pip install -q boto3 openpyxl

In [69]:
import base64
import os
import re
import time
from pathlib import Path

import boto3
import pandas as pd
import requests
from getpass import getpass

## 12. AWS 자격증명 로드
Colab Secrets에 등록돼 있으면 자동으로 읽고, 없으면 안전하게(화면 노출 없이) 직접 입력받습니다.

In [70]:
try:
    from google.colab import userdata
    for k in ("AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_SESSION_TOKEN"):
        try:
            v = userdata.get(k)
            if v:
                os.environ[k] = v
        except Exception:
            pass
except ImportError:
    pass

if not os.environ.get("AWS_ACCESS_KEY_ID") or not os.environ.get("AWS_SECRET_ACCESS_KEY"):
    print("Colab Secrets에 등록된 AWS 자격증명이 없어 직접 입력받습니다.")
    os.environ["AWS_ACCESS_KEY_ID"] = getpass("AWS_ACCESS_KEY_ID 입력: ")
    os.environ["AWS_SECRET_ACCESS_KEY"] = getpass("AWS_SECRET_ACCESS_KEY 입력: ")
    _token = getpass("AWS_SESSION_TOKEN (없으면 Enter): ")
    if _token:
        os.environ["AWS_SESSION_TOKEN"] = _token

print("AWS 자격증명 준비 완료")

AWS 자격증명 준비 완료


## 13. 설정값

In [71]:
AWS_REGION = "us-east-1"
NOVA_PRO_MODEL_ID = "us.amazon.nova-pro-v1:0"

# 이번 판독에 쓸 건수 (Step 01에서 만든 dataset_nova.json 전체 중 앞에서부터 N건)
DATA_EXTRACTION_COUNT = 250

DATASET_PATH = Path("/content/dataset_nova.json")
if not DATASET_PATH.exists():
    DATASET_PATH = OUT_PATH  # Step 01에서 정의한 경로 재사용

MAX_BEFORE_IMAGES = 3
REQUEST_DELAY_SEC = 0.3
TOP_N = 5

## 14. 유틸 함수 (일치 판정)

In [72]:
def normalize_text(s):
    return str(s).strip().replace(" ", "")


def exact_match(a, b):
    if not a or not b:
        return None
    na, nb = normalize_text(a), normalize_text(b)
    return None if (not na or not nb) else na == nb


def partial_match(a, b):
    if not a or not b:
        return None
    na, nb = normalize_text(a), normalize_text(b)
    return None if (not na or not nb) else ((na in nb) or (nb in na))


def rate(series):
    valid = series.dropna()
    n = len(valid)
    if n == 0:
        return 0, 0, 0.0
    c = int((valid == True).sum())  # noqa: E712
    return c, n, round(c / n * 100, 1)

## 15. 이미지 다운로드 · 프롬프트 생성

In [73]:
def download_image(url, timeout=20):
    try:
        resp = requests.get(url, timeout=timeout)
        if resp.status_code == 200:
            return resp.content
        print(f"    이미지 HTTP {resp.status_code}: {url}")
    except Exception as e:
        print(f"    이미지 다운로드 실패: {e}")
    return None


def build_prompt(record):
    import json as _json
    c = record["candidates"]
    return f"""아파트 입주 하자 신고 사진(BEFORE)과 민원 내용을 분석하여 아래 JSON 형식으로만 응답하세요. 실/부위/상세부위/하자원인/공종은 오직 사진과 민원 내용만으로 판단하세요.
마크다운이나 설명 문구 없이 JSON 객체 하나만 출력합니다.

[신고 정보]
- 단지: {record['단지명']}  {record['동']}동 {record['호']}호
- 입주민 민원 내용: {record['고객민원내용'] or '(없음)'}

[용어 구분]
- "실"은 공간 구분(거실/안방/욕실/주방/현관 등), "부위"는 그 공간의 표면/구조 구분(벽/바닥/천장/문 등)입니다.
- "하자원인(하자종류)"은 이 하자가 왜 생겼는지의 분류(예: 시공불량/오염/파손 등)이고,
  "공종"은 이 하자를 보수할 담당 공사 종류입니다. 서로 다른 개념이니 혼동하지 마세요.
- 아래 후보 목록 안에서만 선택하고, 목록에 없는 값을 만들어내지 마세요.

[허용 실 목록]
{_json.dumps(c['실'], ensure_ascii=False)}
[허용 부위 목록]
{_json.dumps(c['부위'], ensure_ascii=False)}
[허용 상세부위 목록]
{_json.dumps(c['상세부위'], ensure_ascii=False)}
[허용 하자원인 목록]
{_json.dumps(c['하자원인'], ensure_ascii=False)}
[허용 공종 목록]
{_json.dumps(c['공종'], ensure_ascii=False)}

[반환 JSON 형식]
{{
  "ai_area": "허용 실 목록 중 하나",
  "ai_part": "허용 부위 목록 중 하나",
  "ai_part_detail": "허용 상세부위 목록 중 하나",
  "ai_defect_cause": "허용 하자원인 목록 중 하나",
  "ai_work_kind": "허용 공종 목록 중 하나",
  "confidence": 0.0~1.0,
  "reason": "판단 근거 한 문장"
}}"""

## 16. JSON 파싱 · Nova Pro 호출 (재시도 포함)

In [74]:
def extract_json(raw):
    try:
        return json.loads(raw.strip())
    except json.JSONDecodeError:
        pass
    m = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", raw, re.DOTALL)
    if m:
        try:
            return json.loads(m.group(1))
        except json.JSONDecodeError:
            pass
    m = re.search(r"\{.*\}", raw, re.DOTALL)
    if m:
        try:
            return json.loads(m.group(0))
        except json.JSONDecodeError:
            pass
    return {"ai_area": None, "ai_part": None, "ai_part_detail": None,
            "ai_defect_cause": None, "ai_work_kind": None, "confidence": None,
            "reason": raw[:200], "parse_error": True}


def invoke_nova(bedrock, record):
    content = []
    used = 0
    for url in record["photos_before_urls"][:MAX_BEFORE_IMAGES]:
        img = download_image(url)
        if img is None:
            continue
        content.append({"image": {"format": "jpeg", "source": {"bytes": base64.standard_b64encode(img).decode()}}})
        used += 1
    if used == 0:
        return {"error": "이미지 다운로드 실패"}, "이미지 없음"

    content.append({"text": build_prompt(record)})
    body = json.dumps({"messages": [{"role": "user", "content": content}],
                        "inferenceConfig": {"maxTokens": 500, "temperature": 0}})

    for attempt in range(1, 5):
        try:
            resp = bedrock.invoke_model(modelId=NOVA_PRO_MODEL_ID, body=body)
            raw_body = json.loads(resp["body"].read())
            raw_text = raw_body["output"]["message"]["content"][0]["text"]
            return extract_json(raw_text), None
        except Exception as e:
            if "Throttling" in type(e).__name__ or "TooManyRequests" in str(e):
                wait = 5 * attempt
                print(f"    [스로틀링] {wait}초 대기 후 재시도")
                time.sleep(wait)
                continue
            return None, str(e)
    return None, "재시도 초과"


## 17. 정답 vs Nova Pro 판독결과 병행 비교표

건별로 **실제값(정답)**과 **Nova Pro 예측값**을 항목마다 나란히 놓고, 일치 여부(✓/✗)를 표시합니다.
전체 건을 한눈에 스캔해서 어디서 틀렸는지 바로 확인할 수 있어요.

In [75]:
def compute_row(record, ai, error, model_id):
    org_area, ai_area = record.get("실"), ai.get("ai_area")
    org_part, ai_part = record.get("부위"), ai.get("ai_part")
    org_detail, ai_detail = record.get("상세부위"), ai.get("ai_part_detail")
    org_cause, ai_cause = record.get("원본_하자원인"), ai.get("ai_defect_cause")
    org_work, ai_work = record.get("원본_공종"), ai.get("ai_work_kind")

    if error:
        status = "오류"
    elif ai.get("parse_error"):
        status = "파싱오류"
    elif ai_work is None:
        status = "미분류"
    else:
        status = "성공"

    return {
        "defect_id": record["defect_id"], "동": record["동"], "호": record["호"],
        "원본_실": org_area, "AI_실": ai_area, "실_일치": exact_match(org_area, ai_area),
        "원본_부위": org_part, "AI_부위": ai_part, "부위_일치": exact_match(org_part, ai_part),
        "원본_상세부위": org_detail, "AI_상세부위": ai_detail, "상세부위_일치": exact_match(org_detail, ai_detail),
        "상세부위_부분일치": partial_match(org_detail, ai_detail),
        "원본_하자원인": org_cause, "AI_하자원인": ai_cause, "하자원인_일치": exact_match(org_cause, ai_cause),
        "원본_공종": org_work, "AI_공종": ai_work, "공종_일치": exact_match(org_work, ai_work),
        "AI_신뢰도": ai.get("confidence"), "AI_판단근거": ai.get("reason"),
        "분석상태": status, "model_used": model_id,
    }

## 18. 정확도 요약 함수 (전체 + 빈도 상위 N개 클래스 기준)

In [76]:
def build_accuracy_summary(df, model_name):
    if df.empty:
        return pd.DataFrame([{"항목": "모델", "값": model_name}, {"항목": "전체 시도 건수", "값": 0}])

    ok = df[df["분석상태"] == "성공"]
    fields = [("실", "실_일치"), ("부위", "부위_일치"),
              ("상세부위_정확일치", "상세부위_일치"), ("상세부위_부분일치", "상세부위_부분일치"),
              ("하자원인", "하자원인_일치"), ("공종", "공종_일치")]
    rows = [{"항목": "모델", "값": model_name},
            {"항목": "전체 시도 건수", "값": len(df)},
            {"항목": "성공 건수", "값": len(ok)}, {"항목": "", "값": ""}]
    for label, col in fields:
        c, n, r = rate(ok[col])
        rows.append({"항목": f"[전체] {label} 일치율", "값": f"{c}/{n} = {r}%"})
    rows.append({"항목": "", "값": ""})
    for label, org_col, match_col in [("부위", "원본_부위", "부위_일치"),
                                       ("하자원인", "원본_하자원인", "하자원인_일치"),
                                       ("공종", "원본_공종", "공종_일치")]:
        valid = ok.dropna(subset=[org_col])
        top_classes = valid[org_col].value_counts().head(TOP_N).index.tolist()
        sub = valid[valid[org_col].isin(top_classes)]
        c, n, r = rate(sub[match_col])
        rows.append({"항목": f"[빈도 상위{TOP_N}] {label} 일치율",
                      "값": f"{c}/{n} = {r}% (대상: {', '.join(map(str, top_classes))})"})
    return pd.DataFrame(rows)

## 19. 실행

In [77]:
def main():
    records = json.loads(DATASET_PATH.read_text(encoding="utf-8"))
    records = records[:DATA_EXTRACTION_COUNT]
    print(f"분석 대상 {len(records)}건 로드 ({DATASET_PATH})")

    bedrock = boto3.client(
        "bedrock-runtime", region_name=AWS_REGION,
        aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
        aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
        aws_session_token=os.environ.get("AWS_SESSION_TOKEN"),
    )

    rows = []
    t0 = time.time()
    for i, record in enumerate(records, 1):
        current_original_detail = record.get("상세부위", "N/A")
        print(f"[{i:>5}/{len(records)}] defect_id={record['defect_id']} 원본_상세부위={current_original_detail}")

        ai, error = invoke_nova(bedrock, record)
        if error and ai is None:
            ai = {"ai_area": None, "ai_part": None, "ai_part_detail": None,
                  "ai_defect_cause": None, "ai_work_kind": None, "error": error}

        current_ai_detail = ai.get("ai_part_detail", "N/A")
        exact_status = " (정확 일치)" if exact_match(current_original_detail, current_ai_detail) else ""
        partial_status = " (부분 일치)" if partial_match(current_original_detail, current_ai_detail) else ""
        print(f"  -> AI_상세부위={current_ai_detail}{exact_status}{partial_status}")

        rows.append(compute_row(record, ai, error, NOVA_PRO_MODEL_ID))
        time.sleep(REQUEST_DELAY_SEC)

    elapsed = time.time() - t0
    df = pd.DataFrame(rows)
    summary = build_accuracy_summary(df, "AWS Bedrock Nova Pro")

    c_exact, n_exact, r_exact = rate(df["상세부위_일치"])
    print(f"\n[전체 요약] 상세부위_정확일치율: {c_exact}/{n_exact} = {r_exact}%")
    c_partial, n_partial, r_partial = rate(df["상세부위_부분일치"])
    print(f"[전체 요약] 상세부위_부분일치율: {c_partial}/{n_partial} = {r_partial}%")

    out_path = Path("NovaPro_분석최종결과.xlsx")
    with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
        summary.to_excel(writer, sheet_name="요약", index=False)
        df.to_excel(writer, sheet_name="전체결과", index=False)
    Path("rows_nova_pro.json").write_text(json.dumps(rows, ensure_ascii=False), encoding="utf-8")

    print(f"\n완료: {len(rows)}건, 소요 {elapsed/60:.1f}분")
    print(f"최종 결과: {out_path.resolve()}")
    return df, summary, rows


df, summary, rows = main()

분석 대상 250건 로드 (/content/dataset_nova.json)
[    1/250] defect_id=13052436 원본_상세부위=타일
  -> AI_상세부위=타일 (정확 일치) (부분 일치)
[    2/250] defect_id=13052450 원본_상세부위=코킹
  -> AI_상세부위=타일
[    3/250] defect_id=13052460 원본_상세부위=벽지
  -> AI_상세부위=도장
[    4/250] defect_id=13052509 원본_상세부위=도장
  -> AI_상세부위=도장 (정확 일치) (부분 일치)
[    5/250] defect_id=13052570 원본_상세부위=도장
  -> AI_상세부위=코킹(내장)
[    6/250] defect_id=13052608 원본_상세부위=빨래건조대스위치
  -> AI_상세부위=미분류
[    7/250] defect_id=13052609 원본_상세부위=엔지니어드스톤
  -> AI_상세부위=타일
[    8/250] defect_id=13052638 원본_상세부위=수납장
  -> AI_상세부위=목문
[    9/250] defect_id=13052667 원본_상세부위=걸레받이
  -> AI_상세부위=코킹
[   10/250] defect_id=13052677 원본_상세부위=천정지
  -> AI_상세부위=벽지
[   11/250] defect_id=13052680 원본_상세부위=디딤판
  -> AI_상세부위=코킹
[   12/250] defect_id=13052731 원본_상세부위=걸레받이
  -> AI_상세부위=코킹
[   13/250] defect_id=13052732 원본_상세부위=스프링클러
  -> AI_상세부위=도장
[   14/250] defect_id=13052753 원본_상세부위=벽지
  -> AI_상세부위=도장
[   15/250] defect_id=13052772 원본_상세부위=목문
  -> AI_상세부위=도장
[   16/250] defect_id=1305285

## 20. 정확도 요약 표

In [78]:
summary

,항목,값
0,모델,AWS Bedrock Nova Pro
1,전체 시도 건수,250
2,성공 건수,243
3,,
4,[전체] 실 일치율,21/243 = 8.6%
5,[전체] 부위 일치율,153/243 = 63.0%
6,[전체] 상세부위_정확일치 일치율,28/243 = 11.5%
7,[전체] 상세부위_부분일치 일치율,34/243 = 14.0%
8,[전체] 하자원인 일치율,98/243 = 40.3%
9,[전체] 공종 일치율,29/242 = 12.0%


## 21. 정답 vs Nova Pro 판독결과 병기 표

`compute_row`에서 이미 `원본_X` / `AI_X`로 병기해뒀으니, 여기서는 보기 좋게 정렬해서 표시하고 CSV로도 저장합니다.

In [79]:
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.max_rows", None)

display_cols = [
    "defect_id", "동", "호",
    "원본_실", "AI_실", "실_일치",
    "원본_부위", "AI_부위", "부위_일치",
    "원본_상세부위", "AI_상세부위", "상세부위_일치", "상세부위_부분일치",
    "원본_하자원인", "AI_하자원인", "하자원인_일치",
    "원본_공종", "AI_공종", "공종_일치",
    "AI_신뢰도", "분석상태",
]
compare_df = df[display_cols]

CSV_PATH = Path("classification_compare.csv")
compare_df.to_csv(CSV_PATH, index=False, encoding="utf-8-sig")
print(f"병기 비교표 저장: {CSV_PATH.resolve()}")

compare_df

병기 비교표 저장: /content/classification_compare.csv


,defect_id,동,호,원본_실,AI_실,실_일치,원본_부위,AI_부위,부위_일치,원본_상세부위,...,상세부위_일치,상세부위_부분일치,원본_하자원인,AI_하자원인,하자원인_일치,원본_공종,AI_공종,공종_일치,AI_신뢰도,분석상태
0,13052436,104,1501,공용욕실,거실,False,벽,벽,True,타일,...,True,True,흠집,시공불량,False,타일공사,타일공사,True,0.95,성공
1,13052450,103,203,주방발코니,욕실,False,벽,바닥,False,코킹,...,False,False,미시공,시공불량,False,코킹공사,타일공사,False,0.95,성공
2,13052460,103,501,침실1,거실,False,벽,벽,True,벽지,...,False,False,시공불량,오염,False,도배공사,도장공사,False,0.90,성공
3,13052509,104,1503,안방전면발코니,공용부,False,천정,천정,True,도장,...,True,True,미시공,시공불량,False,도장공사,도장공사,True,0.95,성공
4,13052570,104,1501,안방전면발코니,거실,False,벽,벽,True,도장,...,False,False,흠집,시공불량,False,도장공사,코킹공사,False,0.95,성공
5,13052608,104,1501,안방전면발코니,거실,False,벽,벽,True,빨래건조대스위치,...,False,False,미시공,미시공,True,가전용품공사,내장공사,False,0.90,성공
6,13052609,109,1404,주방,거실,False,벽,벽,True,엔지니어드스톤,...,False,False,시공불량,시공불량,True,엔지니어드스톤공사,타일공사,False,0.95,성공
7,13052638,103,1503,주방,거실,False,벽,벽,True,수납장,...,False,False,흠집,시공불량,False,주방가구공사,목문공사,False,0.95,성공
8,13052667,104,1403,안방,거실,False,벽,벽,True,걸레받이,...,False,False,시공불량,시공불량,True,내장목공사,코킹공사,False,0.95,성공
9,13052677,109,1104,침실2,복도,False,천정,벽,False,천정지,...,False,False,시공불량,오염,False,도배공사,도배공사,True,0.95,성공


### 21-1. 5개 항목 중 하나라도 불일치한 건만 모아보기

In [80]:
match_cols = ["실_일치", "부위_일치", "상세부위_일치", "하자원인_일치", "공종_일치"]
mismatch_mask = compare_df[match_cols].apply(lambda row: any(v is False for v in row), axis=1)
mismatch_df = compare_df[mismatch_mask]
print(f"불일치 건수: {len(mismatch_df)} / {len(compare_df)}건")
mismatch_df

불일치 건수: 243 / 250건


,defect_id,동,호,원본_실,AI_실,실_일치,원본_부위,AI_부위,부위_일치,원본_상세부위,...,상세부위_일치,상세부위_부분일치,원본_하자원인,AI_하자원인,하자원인_일치,원본_공종,AI_공종,공종_일치,AI_신뢰도,분석상태
0,13052436,104,1501,공용욕실,거실,False,벽,벽,True,타일,...,True,True,흠집,시공불량,False,타일공사,타일공사,True,0.95,성공
1,13052450,103,203,주방발코니,욕실,False,벽,바닥,False,코킹,...,False,False,미시공,시공불량,False,코킹공사,타일공사,False,0.95,성공
2,13052460,103,501,침실1,거실,False,벽,벽,True,벽지,...,False,False,시공불량,오염,False,도배공사,도장공사,False,0.90,성공
3,13052509,104,1503,안방전면발코니,공용부,False,천정,천정,True,도장,...,True,True,미시공,시공불량,False,도장공사,도장공사,True,0.95,성공
4,13052570,104,1501,안방전면발코니,거실,False,벽,벽,True,도장,...,False,False,흠집,시공불량,False,도장공사,코킹공사,False,0.95,성공
5,13052608,104,1501,안방전면발코니,거실,False,벽,벽,True,빨래건조대스위치,...,False,False,미시공,미시공,True,가전용품공사,내장공사,False,0.90,성공
6,13052609,109,1404,주방,거실,False,벽,벽,True,엔지니어드스톤,...,False,False,시공불량,시공불량,True,엔지니어드스톤공사,타일공사,False,0.95,성공
7,13052638,103,1503,주방,거실,False,벽,벽,True,수납장,...,False,False,흠집,시공불량,False,주방가구공사,목문공사,False,0.95,성공
8,13052667,104,1403,안방,거실,False,벽,벽,True,걸레받이,...,False,False,시공불량,시공불량,True,내장목공사,코킹공사,False,0.95,성공
9,13052677,109,1104,침실2,복도,False,천정,벽,False,천정지,...,False,False,시공불량,오염,False,도배공사,도배공사,True,0.95,성공
